## Planning:
- Load data
- Check variable and target
- Check correlation between numeric variable
- Check output to variable impact (using KDE distribution)

In [1]:
import pandas as pd
import numpy as np

In [3]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
submit = pd.read_csv("data/sample_submission.csv")

In [5]:
# check correlation of numeric variable
# take all column numeric column
numeric_cols = train.drop(columns='id').select_dtypes(include='number').columns.tolist()

In [6]:
numeric_cols

['amount_usd',
 'hours_since_last_txn',
 'txn_count_last_24h',
 'distance_from_home_km',
 'card_age_months',
 'customer_age',
 'account_balance_usd',
 'cvv_retry_count',
 'velocity_score',
 'time_of_day_hour',
 'day_of_week',
 'merchant_risk_score',
 'prior_disputes',
 'is_fraud']

In [7]:
numeric_cols_ex = numeric_cols.copy()

In [8]:
numeric_cols_ex.remove("is_fraud")

In [9]:
numeric_cols_ex

['amount_usd',
 'hours_since_last_txn',
 'txn_count_last_24h',
 'distance_from_home_km',
 'card_age_months',
 'customer_age',
 'account_balance_usd',
 'cvv_retry_count',
 'velocity_score',
 'time_of_day_hour',
 'day_of_week',
 'merchant_risk_score',
 'prior_disputes']

In [12]:
bool_cols = train.select_dtypes(include='bool').columns.tolist()

In [13]:
obj_cols = train.select_dtypes(include='object').columns.tolist()

In [14]:
obj_cols

['merchant_category', 'card_type', 'auth_method', 'channel', 'device_type']

In [15]:
train[obj_cols].head(5)

,merchant_category,card_type,auth_method,channel,device_type
0,Groceries,Mastercard,OTP,POS,iPhone
1,Online Retail,Visa,PIN,In-App,Android Phone
2,Utilities,Visa,OTP,Online,Tablet
3,Groceries,Mastercard,3D Secure,POS,Android Phone
4,Electronics,Visa,3D Secure,Online,Windows PC


In [23]:
for col in obj_cols:
    counts = train.groupby(col)["id"].count()
    proportion = counts/counts.sum()

    print(f'col: {col}')
    summary = pd.DataFrame({'count': counts, 'proportion': proportion})
    print(summary)
    print('-'*20)

col: merchant_category
                    count  proportion
merchant_category                    
Crypto Exchange      5285    0.007550
Electronics         48693    0.069565
Fuel                59880    0.085548
Gaming              23784    0.033979
Gift Cards           9456    0.013509
Groceries          142578    0.203694
Healthcare          35889    0.051273
Online Retail      155097    0.221580
Restaurants        101360    0.144808
Streaming           47196    0.067427
Travel              28715    0.041024
Utilities           42027    0.060042
--------------------
col: card_type
             count  proportion
card_type                     
Amex         45797    0.065428
Discover     17235    0.024623
Mastercard  254857    0.364102
RuPay        37070    0.052960
Visa        345001    0.492887
--------------------
col: auth_method
                    count  proportion
auth_method                          
3D Secure          210036    0.300069
Biometric           93666    0.133816
No

In [42]:
# taking sample data for development
train_sample = train.sample(frac=0.01)

In [43]:
train_sample.to_csv("train_sample.csv",index=False)

## Basic processing

For logistic regression, create dummy variable

In [9]:
train_dummy = pd.get_dummies(train, columns= obj_cols,drop_first=True, dtype=int)
test_dummy = pd.get_dummies(test, columns = obj_cols,drop_first=True, dtype=int)

## Model using XGboost

In [21]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score

In [65]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

In [10]:
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, Pool

In [12]:
X = train_dummy.drop(columns=["id","is_fraud"])
y = train_dummy["is_fraud"]
X_test = test_dummy.drop(columns=["id"])

In [83]:
# !pip install lightgbm
# !pip install xgboost
# !pip install catboost

In [24]:
SEED = 42
N_FOLDS = 5
TARGET = 'is_fraud'

In [25]:
# Stratified KFold cross validation
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(skf.split(X, y))

def run_cv(fit_predict_fn, name):
    oof = np.zeros(len(X))
    test_pred = np.zeros(len(X_test))
    scores = []
    for fold, (tr_idx, va_idx) in enumerate(FOLDS):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]
        va_pred, te_pred = fit_predict_fn(X_tr, y_tr, X_va, y_va)
        oof[va_idx] = va_pred
        test_pred += te_pred / N_FOLDS
        auc = roc_auc_score(y_va, va_pred)
        scores.append(auc)
        print(f'  fold {fold}: AUC = {auc:.5f}')
    total = roc_auc_score(y, oof)
    print(f'{name} | OOF AUC = {total:.5f} (mean {np.mean(scores):.5f} ± {np.std(scores):.5f})')
    return oof, test_pred, total

In [86]:
def fit_xgb(X_tr, y_tr, X_va, y_va):
    model = xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='auc',
        n_estimators=5000,
        learning_rate=0.03,
        max_depth=7,
        min_child_weight=20,
        colsample_bytree=0.8,
        subsample=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        tree_method='hist',
        enable_categorical=True,
        early_stopping_rounds=200,
        random_state=SEED,
        n_jobs=-1,
    )
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    return (model.predict_proba(X_va)[:, 1],
            model.predict_proba(X_test)[:, 1])

In [91]:
def save_sub(pred_1,pred_2, name_1,name_2):
    sub = submit.copy()
    sub['is_fraud'] = pred_1
    sub.to_csv(name_1, index=False)
    print('save:', name_1)

    oof_df = train[['id']].copy()
    oof_df.loc[:,name_2] = oof_xgb
    oof_df.to_csv(name_2,index=False)
    print('save:', name_2)

In [87]:
oof_xgb, test_xgb, auc_xgb = run_cv(fit_xgb, 'XGBoost')

  fold 0: AUC = 0.94423
  fold 1: AUC = 0.94858
  fold 2: AUC = 0.95120
  fold 3: AUC = 0.94627
  fold 4: AUC = 0.94157
XGBoost | OOF AUC = 0.94628 (mean 0.94637 ± 0.00334)


In [92]:
save_sub(test_xgb, oof_xgb, 'sub_xgb_2.csv', 'oof_xgb_2.csv')

save: sub_xgb_2.csv
save: oof_xgb_2.csv
